In [ ]:
from pathlib import Path
from langchain_core.tools import tool
import pandas as pd 
from datetime import datetime
project_root=Path.cwd().parent

In [7]:
project_root=Path.cwd().parent
feedbacks_log_path=project_root/"Storage"/"agent_feedbacks.csv"

feedback_log = pd.DataFrame(columns=["timestamp", "intent", "customer_id", "feedback", "rating"])
@tool
def register_feedback(intent: str, customer_id: int, feedback: str, rating: int) -> str:
    """
    Logs customer feedback into the feedback log.

    Args:
        intent (str): The category of the support query (e.g., "cancel_order", "get_refund").
        customer_id (int): The unique ID of the customer.
        feedback (str): The feedback provided by the customer.
        rating(int): The rating provided by the customer out of 5

    Returns:
        str: Success message.
    """
    global feedback_log
    feedback_entry = {
        "timestamp": datetime.now(),
        "intent": intent,
        "customer_id": customer_id,
        "feedback": feedback,
        "rating": rating
    }
    feedback_log = pd.concat([feedback_log, pd.DataFrame([feedback_entry])], ignore_index=True)

    feedback_log.to_csv(feedbacks_log_path)
    return "Feedback registered successfully!"



In [8]:


deferred_cases = pd.DataFrame(columns=["timestamp", "customer_id", "query", "reason","intent"])
deferred_case_path=project_root/"Storage"/"deffered_cases.csv"
@tool
def defer_to_human(customer_id: str, query: str, intent: str, reason: str) -> str:
    """
    Logs customer details and the reason for deferring to a human agent.

    Args:
        customer_id (int): The unique ID of the customer whose query is being deferred.
        query (str): The customer's query or issue that needs human intervention.
        intent (str): The category of the support query (e.g., "order_tracking", "product_description",...etc)
        reason (str): The reason why the query cannot be resolved by the chatbot.

    Returns:
        str: Success message indicating the deferral was logged.
    """
    global deferred_cases
    case_entry = {
        "timestamp": datetime.now(),
        "customer_id": customer_id,
        "query": query,
        "reason": reason,
        "intent": intent
    }
    deferred_cases = pd.concat([deferred_cases, pd.DataFrame([case_entry])], ignore_index=True)
    deferred_cases.to_csv(deferred_case_path, index=False)
    print("defer_to_human success")
    return "Case deferred to human agent and logged successfully!"


@tool
def days_since(delivered_date: str) -> str:
    """
    Calculates the number of days since the product was delivered.

    Args:
        delivered_date (str): The date when the product was delivered in the format 'YYYY-MM-DD'.
    """
    try:
        # Convert the delivered_date string to a datetime object
        delivered_date = datetime.strptime(delivered_date, '%Y-%m-%d')
        today = datetime.today()

        # Calculate the difference in days
        days_difference = str((today - delivered_date).days)

        return days_difference
    except ValueError as e:
        return f"Error: {e}"






In [9]:
# Example Usage
register_feedback.invoke(input={
    "intent": "cancel_order",
    "customer_id": 1,
    "feedback": "Great experience!",
    "rating": 4
})

# Display Feedback Log
print("\nFeedback Log:")
print(feedback_log)



Feedback Log:
                    timestamp        intent customer_id           feedback  \
0  2026-07-25 18:32:39.040921  cancel_order           1  Great experience!   

  rating  
0      4  


In [ ]:

answer=rag.invoke("What is the ram on the mobile phone?")
print(answer)



4GB 

Source: 
product_Smartphone_description.txt


In [12]:
# Example Usage
defer_to_human.invoke(input={
    "customer_id": "12345",
    "query": "I want to speak to a human about my broken item.",
    "intent": "speak_to_agent",
    "reason": "Customer is extremely angry and requested human intervention."
})

# Display Deferred Cases Log
print("\nDeferred Cases Log:")
print(deferred_cases)

defer_to_human success

Deferred Cases Log:
                    timestamp customer_id  \
0  2026-07-25 18:43:40.648806       12345   

                                              query  \
0  I want to speak to a human about my broken item.   

                                              reason          intent  
0  Customer is extremely angry and requested huma...  speak_to_agent  
